<a href="https://colab.research.google.com/github/Vanquan99/Tool_Copy-Google-Drive/blob/main/tool_coppy_driver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Copy Folder / File — Google Drive to Google Drive / Sao chép file hoặc folder giữa các Google Drive
Notebook for Colab to copy a single file or an entire folder between two Google Drives (including shared drives).
Notebook trên Colab để sao chép một file riêng lẻ hoặc toàn bộ folder giữa hai Google Drive (bao gồm shared drive).

## Quick Start / Cách dùng nhanh
1. Open the notebook from the Colab badge or the GitHub link.
1. Mở notebook bằng Colab badge hoặc link GitHub.
2. Run the cells from top to bottom.
2. Chạy lần lượt từng cell từ trên xuống dưới.
3. Paste the source file/folder link or ID and the destination folder link or ID.
3. Dán link hoặc ID của file/folder nguồn và folder đích.
4. If the source is a file, the notebook copies that file directly.
4. Nếu source là file, notebook sẽ copy trực tiếp file đó.

In [ ]:
#@title Input / Nhập thông tin
ui_language = "Tiếng Việt" #@param ["Tiếng Việt", "English"]
# Destination folder link / Link thư mục đích trên Google Drive
dest_link = "" #@param {type:"string"}
# Source file or folder link / Link file hoặc folder nguồn
source_link = "" #@param {type:"string"}
# Start page / Trang bắt đầu, set 0 to start from the first page
from_page = 0 #@param {type:"integer"}
# End page / Trang kết thúc, set 0 to include all pages
to_page = 0 #@param {type:"integer"}
# Maximum size in GB / Giới hạn dung lượng tối đa tính bằng GB
max_download_size_gb = 700 #@param {type:"number"}
# Excluded substrings, comma separated / Chuỗi loại trừ, ngăn cách bằng dấu phẩy
exclude_str = "" #@param {type:"string"}
# Set page values to 0 to process everything / Để 0 ở các ô trang nếu muốn xử lý toàn bộ nội dung

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
service = build('drive', 'v3')
print('Drive service ready')

In [ ]:
import time, re
from datetime import datetime

class DownloadFromDrive:
    MESSAGES = {
        'vi': {
            'start': 'Bắt đầu chạy copy',
            'limit': 'Giới hạn dung lượng',
            'exclude': 'Chuỗi loại trừ',
            'loading': 'Đang lấy thông tin nguồn và đích',
            'source': 'Nguồn',
            'file_copy': 'Nguồn là file, copy trực tiếp',
            'file_done': 'Hoàn tất copy file đơn lẻ',
            'folder_mode': 'Nguồn là folder, sẽ tạo folder tương ứng',
            'done_folder': 'Hoàn tất copy folder',
            'done_run': 'Kết thúc phiên chạy',
            'read_folder': 'Đang đọc nội dung folder',
            'filter': 'Điều kiện lọc',
            'page': 'Trang',
            'items': 'mục',
            'queued': 'Đã thêm trang vào danh sách xử lý',
            'total_items': 'Tổng số mục sẽ xử lý',
            'check_exist': 'Kiểm tra tồn tại',
            'folder_exists': 'Folder đã tồn tại, dùng lại',
            'create_folder': 'Tạo folder mới',
            'folder_created': 'Đã tạo folder',
            'skip_exists': 'Bỏ qua vì file đã tồn tại',
            'copying_file': 'Đang copy file',
            'copied': 'Đã copy xong',
            'total_size': 'Tổng dung lượng đã copy',
            'limit_exceeded': 'Đã vượt giới hạn',
            'enter_folder': 'Bắt đầu vào folder',
            'leave_folder': 'Hoàn tất folder',
            'batch_start': 'Bắt đầu copy',
            'batch_to': 'mục vào folder đích',
            'progress': 'Tiến trình',
            'processed': 'Đã xử lý xong',
        },
        'en': {
            'start': 'Starting copy run',
            'limit': 'Size limit',
            'exclude': 'Excluded strings',
            'loading': 'Loading source and destination info',
            'source': 'Source',
            'file_copy': 'Source is a file, copying directly',
            'file_done': 'Single-file copy completed',
            'folder_mode': 'Source is a folder, creating a matching folder',
            'done_folder': 'Folder copy completed',
            'done_run': 'Run finished',
            'read_folder': 'Reading folder contents',
            'filter': 'Filter query',
            'page': 'Page',
            'items': 'items',
            'queued': 'Added page to processing list',
            'total_items': 'Total items to process',
            'check_exist': 'Checking existence',
            'folder_exists': 'Folder already exists, reusing',
            'create_folder': 'Creating new folder',
            'folder_created': 'Folder created',
            'skip_exists': 'Skipping because file already exists',
            'copying_file': 'Copying file',
            'copied': 'Copied successfully',
            'total_size': 'Total copied size',
            'limit_exceeded': 'Size limit exceeded',
            'enter_folder': 'Entering folder',
            'leave_folder': 'Finished folder',
            'batch_start': 'Starting copy of',
            'batch_to': 'items to destination folder',
            'progress': 'Progress',
            'processed': 'Finished processing',
        },
    }

    def __init__(self, service, ui_language='Tiếng Việt'):
        self.service = service
        self._total_size = 0.0  # MB
        self._limit_size = 0.0  # GB
        self.excluded_strings = []
        self._copied_count = 0
        self.ui_language = 'en' if ui_language == 'English' else 'vi'

    def log(self, message):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def t(self, key):
        return self.MESSAGES.get(self.ui_language, self.MESSAGES['vi']).get(key, key)

    def extract_id(self, value):
        if not value:
            return None
        match = re.search(r'[-\w]{25,}', value)
        return match.group(0) if match else value

    def is_folder(self, file_meta):
        return file_meta.get('mimeType') == 'application/vnd.google-apps.folder'

    def get_children(self, folder_id, from_page=0, to_page=0):
        files = []
        page_token = None
        query = f"'{folder_id}' in parents and trashed = false"
        if self.excluded_strings:
            excluded_query = ' and '.join([f"not name contains '{item}'" for item in self.excluded_strings])
            query += f' and ({excluded_query})'
        self.log(f"{self.t('read_folder')}: {folder_id}")
        self.log(f"{self.t('filter')}: {query}")
        page = 0
        while True:
            response = self.service.files().list(
                q=query,
                orderBy='name, createdTime',
                fields='files(id, name, mimeType, size), nextPageToken',
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True
            ).execute()
            page += 1
            page_items = response.get('files', [])
            self.log(f"{self.t('page')} {page}: {len(page_items)} {self.t('items')}")
            if (from_page < page <= to_page) or to_page == 0:
                files.extend(page_items)
                self.log(f"{self.t('queued')} {page}")
            page_token = response.get('nextPageToken')
            if not page_token or (to_page > 0 and page >= to_page):
                break
        self.log(f"{self.t('total_items')}: {len(files)}")
        return files

    def exists_in_dest(self, dest_folder_id, name):
        try:
            escaped = name.replace("'", "\\'")
            query = f"'{dest_folder_id}' in parents and name = '{escaped}' and trashed = false"
            self.log(f"{self.t('check_exist')}: {name}")
            result = self.service.files().list(q=query, fields='files(id)').execute()
            return bool(result.get('files'))
        except Exception as exc:
            print('exists_in_dest error:', exc)
            return False

    def create_folder(self, dest_folder_id, folder_name):
        if self.exists_in_dest(dest_folder_id, folder_name):
            self.log(f"{self.t('folder_exists')}: {folder_name}")
            return self.service.files().list(
                q=f"'{dest_folder_id}' in parents and name = '{folder_name}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
                fields='files(id)'
            ).execute().get('files', [{}])[0].get('id')
        self.log(f"{self.t('create_folder')}: {folder_name}")
        body = {'name': folder_name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [dest_folder_id]}
        created = self.service.files().create(body=body, fields='id').execute()
        self.log(f"{self.t('folder_created')}: {folder_name} ({created['id']})")
        return created['id']

    def copy_file(self, dest_folder_id, file_meta):
        if self.exists_in_dest(dest_folder_id, file_meta['name']):
            self.log(f"{self.t('skip_exists')}: {file_meta['name']}")
            return
        self._copied_count += 1
        self.log(f"{self.t('copying_file')} {self._copied_count}: {file_meta['name']}")
        start = time.time()
        self.service.files().copy(
            body={'parents': [dest_folder_id]},
            fileId=file_meta['id'],
            supportsAllDrives=True
        ).execute()
        elapsed = time.time() - start
        size_mb = int(file_meta.get('size', 0)) / (1024 * 1024)
        self._total_size += size_mb
        speed = size_mb / elapsed if elapsed > 0 else 0
        self.log(f"{self.t('copied')}: {file_meta['name']} | {size_mb:0.2f} MB | {elapsed:0.2f}s | {speed:0.2f} MB/s")
        self.log(f"{self.t('total_size')}: {self._total_size:0.2f} MB")
        if self._limit_size > 0 and self._total_size >= (self._limit_size * 1024):
            self.on_total_size_exceeded(f"{self.t('limit_exceeded')} {self._limit_size} GB")

    def copy_item(self, dest_folder_id, item_meta):
        if self.is_folder(item_meta):
            self.log(f"{self.t('enter_folder')}: {item_meta['name']}")
            new_folder_id = self.create_folder(dest_folder_id, item_meta['name'])
            children = self.get_children(item_meta['id'], 0, 0)
            self.copy_items(new_folder_id, children)
            self.log(f"{self.t('leave_folder')}: {item_meta['name']}")
        else:
            self.copy_file(dest_folder_id, item_meta)

    def copy_items(self, dest_folder_id, items):
        total = len(items)
        self.log(f"{self.t('batch_start')} {total} {self.t('batch_to')}")
        for index, item in enumerate(items, start=1):
            self.log(f"{self.t('progress')} {index}/{total}: {item.get('name')}")
            self.copy_item(dest_folder_id, item)
        self.log(f"{self.t('processed')} {total} {self.t('items')}")

    def copy_drive_to_drive(self, dest_link, source_link, from_page=0, to_page=0):
        dest_id = self.extract_id(dest_link)
        source_id = self.extract_id(source_link)
        if not dest_id or not source_id:
            raise ValueError('Missing destination or source ID')
        self.log(self.t('loading'))
        source_meta = self.service.files().get(fileId=source_id, supportsAllDrives=True, fields='id,name,mimeType,size').execute()
        self.log(f"{self.t('source')}: {source_meta.get('name')} | Type: {source_meta.get('mimeType')}")
        if not self.is_folder(source_meta):
            self.log(self.t('file_copy'))
            self.copy_file(dest_id, source_meta)
            self.log(self.t('file_done'))
            return
        source_folder_name = source_meta['name']
        self.log(f"{self.t('folder_mode')}: {source_folder_name}")
        target_folder_id = self.create_folder(dest_id, source_folder_name)
        children = self.get_children(source_id, from_page, to_page)
        self.copy_items(target_folder_id, children)
        self.log(f"{self.t('done_folder')}: {source_folder_name}")
        self.log(f"{self.t('total_size')}: {self._total_size:0.2f} MB")

In [ ]:
downloader = DownloadFromDrive(service, ui_language=ui_language)
downloader._limit_size = float(max_download_size_gb)
downloader.excluded_strings = [item.strip() for item in exclude_str.split(',') if item.strip()]
downloader.log(downloader.t('start'))
downloader.log(f"{downloader.t('limit')}: {downloader._limit_size:0.2f} GB")
downloader.log(f"{downloader.t('exclude')}: {downloader.excluded_strings if downloader.excluded_strings else ('Không có' if downloader.ui_language == 'vi' else 'None')}")
downloader.copy_drive_to_drive(dest_link, source_link, from_page, to_page)
downloader.log(downloader.t('done_run'))